# LinkedIn Jobs scraping

This notebook is to scrape last 7 days jobs from LinkedIn. It was run in Aug 2025.

## Scraper 1: Past 7 days for specific titles

This scraped 8000 jobs from past 7 days.


Jobs by Region:
  * sf_bay_area: 2000
  * nyc: 2000
  * austin: 2000
  * seattle: 2000

Final data saved as linkedin_jobs_20250824_055425.csv

In [ ]:
!pip install selenium

In [ ]:
"""
LinkedIn Working Scraper - FINAL VERSION
Based on diagnostic results showing the actual HTML structure
"""

import time
import random
from datetime import datetime
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException, NoSuchElementException
import pandas as pd
import re

class LinkedInWorkingScraper:
    def __init__(self):
        """Initialize scraper with correct selectors from diagnostic"""
        self.driver = None
        self.all_jobs_data = []
        self.seen_job_ids = set()

        # Locations to search
        self.locations = {
            'sf_bay_area': ['San Francisco, CA'],
            'nyc': ['New York, NY'],
            'austin': ['Austin, TX'],
            'seattle': ['Seattle, WA']
        }

        # Tech job search terms
        self.search_terms = [
            'Software Engineer',
            'Senior Software Engineer',
            'Staff Software Engineer',
            'Principal Engineer',
            'Machine Learning Engineer',
            'ML Engineer',
            'AI Engineer',
            'Data Scientist',
            'Data Engineer',
            'DevOps Engineer',
            'Platform Engineer',
            'Backend Engineer',
            'Frontend Engineer',
            'Full Stack Engineer',
            'Site Reliability Engineer',
            'SRE',
            'Security Engineer',
            'Cloud Engineer',
            'Solutions Architect',
            'Engineering Manager'
        ]

    def setup_driver(self, headless=False):
        """Setup Chrome driver"""
        from selenium.webdriver.chrome.options import Options

        chrome_options = Options()
        chrome_options.add_argument("--disable-blink-features=AutomationControlled")
        chrome_options.add_experimental_option("excludeSwitches", ["enable-automation"])
        chrome_options.add_argument("--no-sandbox")
        chrome_options.add_argument("--disable-dev-shm-usage")
        chrome_options.add_argument("--window-size=1920,1080")

        # Disable images for speed
        prefs = {"profile.managed_default_content_settings.images": 2}
        chrome_options.add_experimental_option("prefs", prefs)

        if headless:
            chrome_options.add_argument("--headless=new")

        self.driver = webdriver.Chrome(options=chrome_options)
        print("✅ Driver setup complete")

    def extract_jobs_working(self):
        """
        Extract jobs using the ACTUAL selectors we found in diagnostic
        """
        jobs = []

        try:
            # Wait for content to load
            time.sleep(3)

            # Use the selector that WORKS: base-card__full-link
            job_links = self.driver.find_elements(By.CSS_SELECTOR,
                "a.base-card__full-link[href*='/jobs/view/']")

            print(f"  Found {len(job_links)} job links")

            # Also get all base cards for additional info
            base_cards = self.driver.find_elements(By.CSS_SELECTOR,
                "div.base-search-card.job-search-card")

            print(f"  Found {len(base_cards)} job cards")

            # Process each job link
            for i, link in enumerate(job_links[:50]):  # Limit to 50 per page
                try:
                    job_data = {}

                    # Extract URL and job ID
                    href = link.get_attribute('href')
                    if href:
                        # Extract job ID from URL
                        job_id_match = re.search(r'/view/(\d+)', href)
                        if job_id_match:
                            job_data['job_id'] = job_id_match.group(1)
                        else:
                            job_data['job_id'] = f"job_{i}_{datetime.now().timestamp()}"

                        # Clean URL (remove tracking parameters)
                        clean_url = href.split('?')[0]
                        job_data['url'] = clean_url
                    else:
                        continue

                    # Extract title from link text
                    job_data['title'] = link.text.strip()

                    # Skip if title is empty or looks like UI element
                    if not job_data['title'] or len(job_data['title']) < 5:
                        continue
                    if any(skip in job_data['title'].lower() for skip in
                           ['promoted', 'sign in', 'join now']):
                        continue

                    # Try to get the corresponding base card for more info
                    if i < len(base_cards):
                        card = base_cards[i]

                        # Extract company (in h4 tag within the card)
                        try:
                            company_elem = card.find_element(By.TAG_NAME, "h4")
                            job_data['company'] = company_elem.text.strip()
                        except:
                            job_data['company'] = 'N/A'

                        # Extract location
                        try:
                            location_elem = card.find_element(By.CSS_SELECTOR,
                                ".job-search-card__location")
                            job_data['location'] = location_elem.text.strip()
                        except:
                            job_data['location'] = 'N/A'

                        # Extract posted date
                        try:
                            date_elem = card.find_element(By.CSS_SELECTOR,
                                ".job-search-card__listdate, time")
                            job_data['posted_text'] = date_elem.text.strip()
                        except:
                            job_data['posted_text'] = 'N/A'
                    else:
                        # Fallback if we can't match to a card
                        job_data['company'] = 'N/A'
                        job_data['location'] = 'N/A'
                        job_data['posted_text'] = 'N/A'

                    # Add timestamp
                    job_data['scraped_at'] = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

                    # Add to jobs list
                    jobs.append(job_data)

                    # Show first few for verification
                    if len(jobs) <= 3:
                        print(f"    ✅ {job_data['title']} at {job_data['company']}")

                except Exception as e:
                    continue

            # If first method didn't work well, try H3/H4 pairing as backup
            if len(jobs) < 5:
                print("  Trying H3/H4 extraction as backup...")

                h3_tags = self.driver.find_elements(By.TAG_NAME, "h3")
                h4_tags = self.driver.find_elements(By.TAG_NAME, "h4")

                # Filter out UI elements
                job_h3s = []
                for h3 in h3_tags:
                    text = h3.text.strip()
                    if (text and len(text) > 5 and
                        not any(skip in text.lower() for skip in
                               ['sign', 'join', 'linkedin', 'filter', 'sort'])):
                        job_h3s.append(h3)

                job_h4s = []
                for h4 in h4_tags:
                    text = h4.text.strip()
                    if (text and len(text) > 1 and
                        not any(skip in text.lower() for skip in
                               ['experience', 'type', 'function', 'industry'])):
                        job_h4s.append(h4)

                # Pair them up
                for i in range(min(20, min(len(job_h3s), len(job_h4s)))):
                    job_data = {
                        'job_id': f"h3h4_{i}_{datetime.now().timestamp()}",
                        'title': job_h3s[i].text.strip(),
                        'company': job_h4s[i].text.strip() if i < len(job_h4s) else 'N/A',
                        'location': 'N/A',
                        'posted_text': 'N/A',
                        'url': 'N/A',
                        'scraped_at': datetime.now().strftime("%Y-%m-%d %H:%M:%S")
                    }

                    # Check if not duplicate
                    if job_data['title'] not in [j['title'] for j in jobs]:
                        jobs.append(job_data)

            print(f"  📊 Extracted {len(jobs)} jobs from page")

        except Exception as e:
            print(f"  ❌ Error in extraction: {str(e)}")

        return jobs

    def search_jobs(self, search_term, location, posted_days=7):
        """Perform job search"""
        base_url = "https://www.linkedin.com/jobs/search?"
        params = [
            f"keywords={search_term.replace(' ', '%20')}",
            f"location={location.replace(' ', '%20').replace(',', '%2C')}"
        ]

        # Time filter
        if posted_days <= 1:
            params.append("f_TPR=r86400")
        elif posted_days <= 7:
            params.append("f_TPR=r604800")
        elif posted_days <= 30:
            params.append("f_TPR=r2592000")

        params.append("sortBy=DD")  # Sort by date

        search_url = base_url + "&".join(params)
        print(f"\n🔍 Searching: {search_term} in {location}")

        self.driver.get(search_url)

        # Wait for content to load
        time.sleep(random.uniform(4, 6))

        # Wait for job cards to appear
        try:
            WebDriverWait(self.driver, 10).until(
                EC.presence_of_element_located((By.CSS_SELECTOR, "a.base-card__full-link"))
            )
        except:
            print("  ⚠️ Timeout waiting for job cards")

        # Check results count
        try:
            title = self.driver.title
            if "jobs" in title.lower():
                match = re.search(r'([\d,]+)\+?\s+', title)
                if match:
                    count = match.group(1)
                    print(f"  Found {count}+ jobs")
        except:
            pass

        return search_url

    def scrape_multiple_pages(self, max_pages=2):
        """Scrape multiple pages of results"""
        all_jobs = []

        for page in range(max_pages):
            print(f"  📄 Page {page + 1}...")

            if page > 0:
                # Navigate to next page
                try:
                    # Scroll to bottom
                    self.driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
                    time.sleep(2)

                    # Update URL for pagination
                    current_url = self.driver.current_url
                    if "start=" in current_url:
                        new_url = re.sub(r'start=\d+', f'start={page*25}', current_url)
                    else:
                        separator = "&" if "?" in current_url else "?"
                        new_url = f"{current_url}{separator}start={page*25}"

                    self.driver.get(new_url)
                    time.sleep(random.uniform(3, 5))

                    # Wait for new content
                    try:
                        WebDriverWait(self.driver, 10).until(
                            EC.presence_of_element_located((By.CSS_SELECTOR, "a.base-card__full-link"))
                        )
                    except:
                        print("    No more pages available")
                        break
                except:
                    break

            # Extract jobs
            page_jobs = self.extract_jobs_working()

            # Deduplicate
            for job in page_jobs:
                if job['job_id'] not in self.seen_job_ids:
                    self.seen_job_ids.add(job['job_id'])
                    all_jobs.append(job)

            if len(page_jobs) == 0:
                print("  No more jobs found")
                break

        return all_jobs

    def run_full_scrape(self, max_searches=None, max_pages_per_search=2):
        """Run the complete scraping process"""
        if max_searches is None:
            max_searches = len(self.locations) * len(self.search_terms)

        print("\n" + "="*60)
        print("STARTING FULL SCRAPE")
        print(f"Regions: {len(self.locations)}")
        print(f"Search terms: {len(self.search_terms)}")
        print(f"Max searches: {max_searches}")
        print("="*60)

        all_jobs = []
        search_count = 0

        for region_name, cities in self.locations.items():
            print(f"\n📍 REGION: {region_name.upper()}")
            region_jobs = []

            for city in cities:
                for search_term in self.search_terms:
                    if search_count >= max_searches:
                        print(f"\n⚠️ Reached max searches ({max_searches})")
                        break

                    search_count += 1
                    print(f"\n[{search_count}/{max_searches}]", end=" ")

                    # Perform search
                    self.search_jobs(search_term, city)

                    # Scrape results
                    jobs = self.scrape_multiple_pages(max_pages_per_search)

                    # Add metadata
                    for job in jobs:
                        job['search_term'] = search_term
                        job['search_location'] = city
                        job['search_region'] = region_name

                        # Use search location if location not found
                        if job['location'] == 'N/A':
                            job['location'] = city

                    region_jobs.extend(jobs)
                    all_jobs.extend(jobs)

                    print(f"  Total unique jobs: {len(self.seen_job_ids)}")

                    # Save progress
                    if len(all_jobs) > 0 and len(all_jobs) % 100 == 0:
                        self.save_results(all_jobs, interim=True)

                    # Random delay
                    delay = random.uniform(5, 10)
                    time.sleep(delay)

                    # Occasional longer break
                    if search_count % 10 == 0:
                        print("  Taking longer break...")
                        time.sleep(random.uniform(15, 25))

                if search_count >= max_searches:
                    break

            print(f"\n✅ {region_name}: Found {len(region_jobs)} jobs")

            if search_count >= max_searches:
                break

        self.all_jobs_data = all_jobs
        return all_jobs

    def save_results(self, jobs, interim=False):
        """Save results to CSV"""
        if not jobs:
            print("No jobs to save")
            return None

        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        prefix = "interim_" if interim else ""
        filename = f"linkedin_jobs_{prefix}{timestamp}.csv"

        df = pd.DataFrame(jobs)
        df = df.drop_duplicates(subset=['job_id'], keep='first')

        # Clean data
        if 'company' in df.columns:
            df['company'] = df['company'].str.strip()
        if 'title' in df.columns:
            df['title'] = df['title'].str.strip()

        df.to_csv(filename, index=False)

        if not interim:
            print(f"\n💾 Saved {len(df)} unique jobs to {filename}")

            # Analysis
            print(f"\n📊 SUMMARY:")
            print(f"  Total unique jobs: {len(df)}")

            if 'company' in df.columns:
                with_company = len(df[df['company'] != 'N/A'])
                print(f"  Jobs with company: {with_company} ({with_company/len(df)*100:.1f}%)")

                print(f"\n  Top 20 Companies:")
                top_companies = df[df['company'] != 'N/A']['company'].value_counts().head(20)
                for company, count in top_companies.items():
                    print(f"    {company}: {count}")

            if 'search_region' in df.columns:
                print(f"\n  Jobs by Region:")
                for region, count in df['search_region'].value_counts().items():
                    print(f"    {region}: {count}")

        return filename

    def close(self):
        """Close browser"""
        if self.driver:
            self.driver.quit()
            print("✅ Browser closed")


# Main execution
def run_working_scraper():
    """Run the working scraper in Colab"""
    print("="*60)
    print("LINKEDIN WORKING SCRAPER")
    print("Using selectors confirmed by diagnostic")
    print("="*60)

    # Setup
    import subprocess
    import sys

    print("📦 Installing requirements...")
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'selenium', 'pandas'], capture_output=True)

    print("🔧 Setting up Chrome driver...")
    subprocess.run(['apt-get', 'update'], capture_output=True)
    subprocess.run(['apt', 'install', 'chromium-chromedriver'], capture_output=True)
    sys.path.insert(0, '/usr/lib/chromium-browser/chromedriver')

    # Initialize scraper
    scraper = LinkedInWorkingScraper()

    # Configuration
    MAX_SEARCHES = 80  # Adjust as needed
    PAGES_PER_SEARCH = 2

    print(f"\n⚙️ Configuration:")
    print(f"  Regions: {list(scraper.locations.keys())}")
    print(f"  Search terms: {len(scraper.search_terms)}")
    print(f"  Max searches: {MAX_SEARCHES}")
    print(f"  Pages per search: {PAGES_PER_SEARCH}")
    print(f"  Estimated time: {MAX_SEARCHES * 0.5:.0f}-{MAX_SEARCHES:.0f} minutes")

    try:
        scraper.setup_driver(headless=True)

        print("\n🚀 Starting scrape with working selectors...")
        jobs = scraper.run_full_scrape(
            max_searches=MAX_SEARCHES,
            max_pages_per_search=PAGES_PER_SEARCH
        )

        if jobs:
            filename = scraper.save_results(jobs)

            print("\n" + "="*60)
            print(f"✅ SUCCESS! Scraped {len(scraper.seen_job_ids)} unique jobs")
            print("="*60)

            # Download
            from google.colab import files
            if filename:
                files.download(filename)
                print(f"📥 Downloaded: {filename}")
        else:
            print("\n⚠️ No jobs extracted")

    except Exception as e:
        print(f"\n❌ Error: {e}")
        import traceback
        traceback.print_exc()

    finally:
        scraper.close()
        print("\n🏁 Complete!")


if __name__ == "__main__":
    try:
        import google.colab
        run_working_scraper()
    except ImportError:
        print("Run this in Google Colab")

# Analysis of first scraper

In [5]:
"""
LinkedIn Jobs Market Analysis
For dataset from targeted scraper with specific job titles
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter, defaultdict
import re
from datetime import datetime

class LinkedInJobAnalyzer:
    def __init__(self, csv_path):
        """Initialize with path to scraped CSV"""
        self.csv_path = csv_path
        self.df = None

    def load_data(self):
        """Load and inspect the scraped data"""
        try:
            self.df = pd.read_csv(self.csv_path)
            print(f"✓ Loaded {len(self.df):,} jobs")

            # Show columns
            print(f"\nColumns in dataset:")
            for col in self.df.columns:
                null_count = self.df[col].isnull().sum()
                null_pct = null_count / len(self.df) * 100
                print(f"  - {col}: {len(self.df) - null_count:,} values ({null_pct:.1f}% null)")

            return True
        except Exception as e:
            print(f"✗ Error loading data: {e}")
            return False

    def clean_data(self):
        """Clean and standardize the data"""
        print("\n" + "="*60)
        print("DATA CLEANING")
        print("="*60)

        # Remove duplicates
        original_count = len(self.df)
        self.df = self.df.drop_duplicates(subset=['job_id'], keep='first')
        print(f"Removed {original_count - len(self.df)} duplicate jobs")

        # Clean company names
        if 'company' in self.df.columns:
            self.df['company'] = self.df['company'].str.strip()
            self.df['company'] = self.df['company'].replace('', 'N/A')
            self.df['company'] = self.df['company'].fillna('N/A')

        # Clean titles
        if 'title' in self.df.columns:
            self.df['title'] = self.df['title'].str.strip()
            # Remove "Hiring multiple candidates" suffix
            self.df['title'] = self.df['title'].str.replace(r'\s*-?\s*Hiring multiple candidates?$', '', regex=True)

        # Standardize locations
        if 'location' in self.df.columns:
            self.df['location'] = self.df['location'].str.strip()
            self.df['location'] = self.df['location'].fillna('N/A')

    def extract_seniority(self):
        """Extract seniority level from title"""
        def get_seniority(title):
            if pd.isna(title):
                return 'Unknown'

            title_lower = title.lower()

            # Order matters - check most specific first
            if any(term in title_lower for term in ['intern', 'internship']):
                return 'Intern'
            elif any(term in title_lower for term in ['chief', 'cto', 'cfo', 'coo', 'ceo', 'cpo']):
                return 'C-Suite'
            elif 'vp' in title_lower or 'vice president' in title_lower:
                return 'VP'
            elif 'director' in title_lower:
                return 'Director'
            elif 'head of' in title_lower:
                return 'Head'
            elif any(term in title_lower for term in ['principal', 'distinguished']):
                return 'Principal'
            elif 'staff' in title_lower:
                return 'Staff'
            elif 'senior' in title_lower or 'sr.' in title_lower or 'sr ' in title_lower:
                return 'Senior'
            elif 'lead' in title_lower and 'leader' not in title_lower:
                return 'Lead'
            elif any(term in title_lower for term in ['junior', 'jr.', 'jr ', 'entry level',
                                                      'new grad', 'early career']):
                return 'Junior'
            elif 'manager' in title_lower:
                return 'Manager'
            elif any(term in title_lower for term in ['engineer', 'developer', 'scientist',
                                                      'analyst', 'architect']):
                # Default IC roles without modifier
                return 'Mid-Level'
            else:
                return 'Other'

        self.df['seniority'] = self.df['title'].apply(get_seniority)

    def extract_role_type(self):
        """Extract the core role type from title"""
        def get_role_type(title):
            if pd.isna(title):
                return 'Unknown'

            title_lower = title.lower()

            # Management track
            if 'manager' in title_lower and 'program' not in title_lower and 'product' not in title_lower:
                return 'Engineering Manager'

            # Specific engineering roles (check before generic)
            if any(term in title_lower for term in ['machine learning', 'ml engineer', 'ai engineer']):
                return 'ML/AI Engineer'
            elif 'data scientist' in title_lower:
                return 'Data Scientist'
            elif 'data engineer' in title_lower:
                return 'Data Engineer'
            elif any(term in title_lower for term in ['devops', 'site reliability', 'sre']):
                return 'DevOps/SRE'
            elif 'platform engineer' in title_lower:
                return 'Platform Engineer'
            elif any(term in title_lower for term in ['backend', 'back-end']):
                return 'Backend Engineer'
            elif any(term in title_lower for term in ['frontend', 'front-end']):
                return 'Frontend Engineer'
            elif any(term in title_lower for term in ['full stack', 'fullstack']):
                return 'Full Stack Engineer'
            elif 'security engineer' in title_lower:
                return 'Security Engineer'
            elif 'cloud engineer' in title_lower:
                return 'Cloud Engineer'
            elif any(term in title_lower for term in ['solutions architect', 'solution architect']):
                return 'Solutions Architect'
            elif 'software engineer' in title_lower:
                return 'Software Engineer'
            else:
                return 'Other'

        self.df['role_type'] = self.df['title'].apply(get_role_type)

    def analyze_search_effectiveness(self):
        """Analyze which search terms found what"""
        print("\n" + "="*60)
        print("SEARCH TERM EFFECTIVENESS")
        print("="*60)

        if 'search_term' not in self.df.columns:
            print("No search_term column found")
            return

        # Jobs per search term
        search_counts = self.df['search_term'].value_counts()

        print("\nJobs found by search term:")
        for term, count in search_counts.head(20).items():
            pct = count / len(self.df) * 100
            print(f"  {term:30} {count:4} jobs ({pct:5.1f}%)")

        # What did each search term actually find?
        print("\n" + "-"*40)
        print("SEARCH ACCURACY (What we searched vs what we got)")
        print("-"*40)

        for search_term in search_counts.head(10).index:
            term_df = self.df[self.df['search_term'] == search_term]

            # Check if titles match search term
            matching = term_df[term_df['title'].str.contains(search_term, case=False, na=False)]
            match_rate = len(matching) / len(term_df) * 100

            print(f"\n'{search_term}':")
            print(f"  Found {len(term_df)} jobs, {match_rate:.1f}% contain exact term")

            # Show what we actually got
            actual_roles = term_df['role_type'].value_counts().head(3)
            print(f"  Top actual roles found:")
            for role, count in actual_roles.items():
                print(f"    - {role}: {count} ({count/len(term_df)*100:.1f}%)")

    def analyze_market_demand(self):
        """Analyze overall market demand"""
        print("\n" + "="*60)
        print("MARKET DEMAND ANALYSIS")
        print("="*60)

        # Overall stats
        print(f"\nDataset Overview:")
        print(f"  Total jobs: {len(self.df):,}")
        print(f"  Unique companies: {self.df['company'].nunique():,}")
        print(f"  Unique job titles: {self.df['title'].nunique():,}")

        # Geographic distribution
        if 'search_region' in self.df.columns:
            print(f"\nGeographic Distribution:")
            for region, count in self.df['search_region'].value_counts().items():
                pct = count / len(self.df) * 100
                print(f"  {region:15} {count:5} jobs ({pct:5.1f}%)")

        # Role type distribution
        print(f"\nRole Type Distribution:")
        role_counts = self.df['role_type'].value_counts()
        for role, count in role_counts.head(15).items():
            pct = count / len(self.df) * 100
            bar = '█' * int(pct/2)
            print(f"  {role:25} {count:4} ({pct:4.1f}%) {bar}")

    def analyze_seniority_distribution(self):
        """Analyze seniority requirements"""
        print("\n" + "="*60)
        print("SENIORITY ANALYSIS")
        print("="*60)

        seniority_counts = self.df['seniority'].value_counts()

        # Group into categories
        leadership = ['C-Suite', 'VP', 'Director', 'Head', 'Manager']
        senior_ic = ['Principal', 'Staff', 'Senior', 'Lead']
        junior_ic = ['Junior', 'Intern']
        mid_ic = ['Mid-Level']

        leadership_count = self.df[self.df['seniority'].isin(leadership)].shape[0]
        senior_ic_count = self.df[self.df['seniority'].isin(senior_ic)].shape[0]
        junior_ic_count = self.df[self.df['seniority'].isin(junior_ic)].shape[0]
        mid_ic_count = self.df[self.df['seniority'].isin(mid_ic)].shape[0]

        print("\nSeniority Groups:")
        print(f"  Leadership (C-Suite/VP/Director/Manager): {leadership_count:,} ({leadership_count/len(self.df)*100:.1f}%)")
        print(f"  Senior IC (Principal/Staff/Senior/Lead): {senior_ic_count:,} ({senior_ic_count/len(self.df)*100:.1f}%)")
        print(f"  Mid-Level IC: {mid_ic_count:,} ({mid_ic_count/len(self.df)*100:.1f}%)")
        print(f"  Junior IC (Junior/Intern): {junior_ic_count:,} ({junior_ic_count/len(self.df)*100:.1f}%)")

        print("\nDetailed Breakdown:")
        for level, count in seniority_counts.items():
            pct = count / len(self.df) * 100
            print(f"  {level:15} {count:5} jobs ({pct:5.1f}%)")

        # Seniority by role type
        print("\n" + "-"*40)
        print("SENIORITY BY ROLE TYPE")
        print("-"*40)

        top_roles = self.df['role_type'].value_counts().head(5).index
        for role in top_roles:
            role_df = self.df[self.df['role_type'] == role]
            print(f"\n{role}:")
            for level, count in role_df['seniority'].value_counts().head(5).items():
                pct = count / len(role_df) * 100
                print(f"  {level:12} {count:3} jobs ({pct:4.1f}%)")

    def analyze_companies(self):
        """Analyze hiring companies"""
        print("\n" + "="*60)
        print("TOP HIRING COMPANIES")
        print("="*60)

        # Remove N/A
        df_clean = self.df[self.df['company'] != 'N/A'].copy()

        company_counts = df_clean['company'].value_counts().head(30)

        print("\nTop 30 companies by job postings:")
        for i, (company, count) in enumerate(company_counts.items(), 1):
            # What are they hiring for?
            company_df = df_clean[df_clean['company'] == company]
            top_role = company_df['role_type'].value_counts().index[0] if len(company_df) > 0 else 'Unknown'
            top_seniority = company_df['seniority'].value_counts().index[0] if len(company_df) > 0 else 'Unknown'

            print(f"{i:2}. {company:30} {count:3} jobs (mainly {top_seniority} {top_role})")

    def find_opportunities(self):
        """Identify market opportunities and gaps"""
        print("\n" + "="*60)
        print("MARKET OPPORTUNITIES & INSIGHTS")
        print("="*60)

        # Junior opportunities by role
        print("\nJunior/Entry-Level Opportunities:")
        junior_df = self.df[self.df['seniority'].isin(['Junior', 'Intern'])]

        if len(junior_df) > 0:
            print(f"Total junior positions: {len(junior_df)} ({len(junior_df)/len(self.df)*100:.1f}% of market)")
            print("\nBy role type:")
            for role, count in junior_df['role_type'].value_counts().head(10).items():
                print(f"  {role:25} {count:3} junior positions")
        else:
            print("No junior positions found in dataset")

        # High demand roles (most postings)
        print("\n" + "-"*40)
        print("HIGHEST DEMAND ROLES")
        print("-"*40)

        # Combine role type and seniority for specific opportunities
        self.df['role_seniority'] = self.df['seniority'] + ' ' + self.df['role_type']
        top_combinations = self.df['role_seniority'].value_counts().head(20)

        print("\nTop 20 role+seniority combinations:")
        for combo, count in top_combinations.items():
            pct = count / len(self.df) * 100
            print(f"  {combo:40} {count:4} jobs ({pct:4.1f}%)")

        # Geographic opportunities
        if 'search_region' in self.df.columns:
            print("\n" + "-"*40)
            print("GEOGRAPHIC OPPORTUNITIES")
            print("-"*40)

            for region in self.df['search_region'].unique():
                region_df = self.df[self.df['search_region'] == region]
                print(f"\n{region}:")
                print(f"  Total jobs: {len(region_df)}")

                # Top role in this region
                top_role = region_df['role_type'].value_counts().index[0]
                top_role_count = region_df['role_type'].value_counts().values[0]
                print(f"  Top role: {top_role} ({top_role_count} jobs)")

                # Junior friendliness
                junior_count = len(region_df[region_df['seniority'].isin(['Junior', 'Intern'])])
                print(f"  Junior positions: {junior_count} ({junior_count/len(region_df)*100:.1f}%)")

    def generate_insights(self):
        """Generate actionable insights"""
        print("\n" + "="*60)
        print("KEY INSIGHTS")
        print("="*60)

        # Calculate key metrics
        total_jobs = len(self.df)

        # Seniority insights
        junior_pct = len(self.df[self.df['seniority'].isin(['Junior', 'Intern'])]) / total_jobs * 100
        senior_pct = len(self.df[self.df['seniority'].isin(['Senior', 'Staff', 'Principal'])]) / total_jobs * 100
        management_pct = len(self.df[self.df['seniority'].isin(['Manager', 'Director', 'VP', 'C-Suite'])]) / total_jobs * 100

        # Role insights
        swe_pct = len(self.df[self.df['role_type'] == 'Software Engineer']) / total_jobs * 100
        ml_pct = len(self.df[self.df['role_type'] == 'ML/AI Engineer']) / total_jobs * 100

        print(f"""
Based on {total_jobs:,} job postings:

1. SENIORITY REALITY:
   - Junior/Entry: {junior_pct:.1f}% of market
   - Senior/Staff/Principal: {senior_pct:.1f}% of market
   - Management: {management_pct:.1f}% of market

2. ROLE DEMAND:
   - Software Engineer: {swe_pct:.1f}%
   - ML/AI Engineer: {ml_pct:.1f}%
   - Top 3 roles: {', '.join(self.df['role_type'].value_counts().head(3).index)}

3. COMPANY INSIGHTS:
   - Total unique companies: {self.df['company'].nunique():,}
   - Top hiring company: {self.df['company'].value_counts().index[0]} ({self.df['company'].value_counts().values[0]} jobs)

4. NOTABLE PATTERNS:
   - Most common title pattern: {self.df['seniority'].value_counts().index[0]} {self.df['role_type'].value_counts().index[0]}
   - Least common seniority: {self.df['seniority'].value_counts().index[-1]} ({self.df['seniority'].value_counts().values[-1]} jobs)
        """)

        # Specific recommendations
        print("\n" + "-"*40)
        print("RECOMMENDATIONS")
        print("-"*40)

        if junior_pct < 10:
            print("⚠️  Junior market is extremely tight - consider upskilling to mid-level")

        if management_pct > 20:
            print("📈 Strong management demand - IC professionals should consider leadership track")

        if ml_pct < 10:
            print("🤖 ML/AI is less than 10% of market - traditional SWE skills still dominant")

        # Best opportunities
        print("\n🎯 BEST OPPORTUNITIES (high demand, multiple openings):")
        top_5 = self.df['role_seniority'].value_counts().head(5)
        for combo, count in top_5.items():
            if count > 20:
                print(f"   - {combo}: {count} openings")

    def save_enhanced_dataset(self):
        """Save the enhanced dataset with new columns"""
        output_file = 'linkedin_jobs_enhanced.csv'
        self.df.to_csv(output_file, index=False)
        print(f"\n💾 Enhanced dataset saved to: {output_file}")
        return output_file

    def run_full_analysis(self):
        """Run the complete analysis pipeline"""
        if not self.load_data():
            return None

        print("\n" + "="*60)
        print("LINKEDIN JOB MARKET ANALYSIS")
        print(f"Analyzing {len(self.df):,} job postings")
        print("="*60)

        # Clean and enhance data
        self.clean_data()
        self.extract_seniority()
        self.extract_role_type()

        # Run analyses
        self.analyze_search_effectiveness()
        self.analyze_market_demand()
        self.analyze_seniority_distribution()
        self.analyze_companies()
        self.find_opportunities()
        self.generate_insights()

        # Save enhanced dataset
        self.save_enhanced_dataset()

        return self.df


# Usage
if __name__ == "__main__":
    # Use the file from Google Drive
    analyzer = LinkedInJobAnalyzer('/content/drive/MyDrive/linkedin_jobs_20250824_055425.csv')
    df_analyzed = analyzer.run_full_analysis()

✓ Loaded 8,000 jobs

Columns in dataset:
  - job_id: 8,000 values (0.0% null)
  - url: 8,000 values (0.0% null)
  - title: 8,000 values (0.0% null)
  - company: 8,000 values (0.0% null)
  - location: 8,000 values (0.0% null)
  - posted_text: 8,000 values (0.0% null)
  - scraped_at: 8,000 values (0.0% null)
  - search_term: 8,000 values (0.0% null)
  - search_location: 8,000 values (0.0% null)
  - search_region: 8,000 values (0.0% null)

LINKEDIN JOB MARKET ANALYSIS
Analyzing 8,000 job postings

DATA CLEANING
Removed 0 duplicate jobs

SEARCH TERM EFFECTIVENESS

Jobs found by search term:
  Software Engineer               400 jobs (  5.0%)
  Senior Software Engineer        400 jobs (  5.0%)
  Staff Software Engineer         400 jobs (  5.0%)
  Principal Engineer              400 jobs (  5.0%)
  Machine Learning Engineer       400 jobs (  5.0%)
  ML Engineer                     400 jobs (  5.0%)
  AI Engineer                     400 jobs (  5.0%)
  Data Scientist                  400 jobs

# Scraper 2

This scraper was sampled properly to not overrepresent certain jobs. Saved 8316 jobs. Saved to Drive: /content/drive/MyDrive/linkedin_scraper_data/session_20250825_143043/final_jobs_20250825_235526.csv.

## LinkedIn Job Scraper with Auto-Save

**Purpose:** Scrapes tech job listings from LinkedIn with automatic Google Drive backup to prevent data loss.

**Key Features:**
- **Stratified sampling** across 5 major tech hubs (SF Bay Area, Seattle, NYC, Austin, Boston) with weighted distribution
- **Auto-checkpoint system** saves progress every 5 minutes to Google Drive
- **Resume capability** - can restart from last checkpoint if interrupted
- **Deduplication** prevents collecting the same job multiple times
- **Smart categorization** classifies jobs into categories (AI/ML, Data Science, Frontend, Backend, etc.)

**Data Collection:**
- Searches across multiple terms, locations, and experience levels
- Extracts: job title, company, location, posting date, and URL
- Targets 10,000 jobs by default with configurable sampling weights
- Uses randomized search sequences to avoid detection

**Output:** CSV files with complete job data, plus JSON summaries with statistics saved to both Google Drive and local backup.

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
"""
Bulletproof Tech Job Demand Scraper with Google Drive Auto-Save
Never lose data again - saves progressively to Google Drive
"""

import time
import random
from datetime import datetime, timedelta
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException, NoSuchElementException
import pandas as pd
import numpy as np
import re
import json
from collections import defaultdict
from scipy import stats
import os
import pickle

class BulletproofDemandScraper:
    def __init__(self, drive_folder='scraper_data'):
        """
        Initialize scraper with Google Drive backup

        Args:
            drive_folder: Folder name in Google Drive for saving data
        """
        self.driver = None
        self.all_jobs_data = []
        self.seen_job_ids = set()
        self.search_metadata = defaultdict(lambda: defaultdict(int))

        # CRITICAL: Setup Google Drive paths
        self.drive_base = '/content/drive/MyDrive'
        self.drive_folder = f'{self.drive_base}/{drive_folder}'
        self.session_id = datetime.now().strftime("%Y%m%d_%H%M%S")
        self.session_folder = f'{self.drive_folder}/session_{self.session_id}'

        # Create folders if they don't exist
        self.setup_drive_folders()

        # Setup checkpoint system
        self.checkpoint_file = f'{self.session_folder}/checkpoint.pkl'
        self.jobs_backup_file = f'{self.session_folder}/jobs_incremental.csv'
        self.last_checkpoint_time = time.time()
        self.checkpoint_interval = 300  # Save every 5 minutes

        # Load any existing checkpoint (for resume capability)
        self.load_checkpoint()

        # [Rest of your existing config - search terms, cities, etc.]
        self.generic_search_terms = [
            'software', 'data', 'machine learning', 'ML', 'AI',
            'artificial intelligence', 'product manager', 'technical program',
            'cloud', 'platform', 'backend', 'frontend', 'full stack',
            'mobile', 'devops', 'security', 'infrastructure',
            'developer', 'programmer', 'scientist', 'architect', 'engineering'
        ]

        self.cities_config = {
            'sf_bay_area': {
                'locations': ['San Francisco, CA', 'San Jose, CA', 'Palo Alto, CA'],
                'weight': 0.35,
                'name': 'SF Bay Area'
            },
            'seattle': {
                'locations': ['Seattle, WA'],
                'weight': 0.20,
                'name': 'Seattle'
            },
            'nyc': {
                'locations': ['New York, NY'],
                'weight': 0.20,
                'name': 'New York'
            },
            'austin': {
                'locations': ['Austin, TX'],
                'weight': 0.15,
                'name': 'Austin'
            },
            'boston': {
                'locations': ['Boston, MA'],
                'weight': 0.10,
                'name': 'Boston'
            }
        }

        self.time_filter = 'r604800'  # Last 7 days
        self.experience_levels = [
            {'name': 'entry_level', 'param': 'f_E=2'},
            {'name': 'associate', 'param': 'f_E=3'},
            {'name': 'mid_senior', 'param': 'f_E=4'},
            {'name': 'director', 'param': 'f_E=5'},
            {'name': 'executive', 'param': 'f_E=6'}
        ]

    def setup_drive_folders(self):
        """Create necessary folders in Google Drive"""
        try:
            # Create main folder
            os.makedirs(self.drive_folder, exist_ok=True)
            # Create session folder
            os.makedirs(self.session_folder, exist_ok=True)
            print(f"✅ Google Drive folders created: {self.session_folder}")
        except Exception as e:
            print(f"⚠️ Error creating folders: {e}")
            print("Continuing with local storage as backup...")

    def save_checkpoint(self):
        """Save current state to Google Drive"""
        checkpoint_data = {
            'all_jobs_data': self.all_jobs_data,
            'seen_job_ids': self.seen_job_ids,
            'search_metadata': dict(self.search_metadata),
            'timestamp': datetime.now().isoformat()
        }

        try:
            # Save checkpoint
            with open(self.checkpoint_file, 'wb') as f:
                pickle.dump(checkpoint_data, f)

            # Also save jobs as CSV for easy access
            if self.all_jobs_data:
                df = pd.DataFrame(self.all_jobs_data)
                df.to_csv(self.jobs_backup_file, index=False)

            print(f"    💾 Checkpoint saved ({len(self.all_jobs_data)} jobs)")

        except Exception as e:
            print(f"    ⚠️ Checkpoint save failed: {e}")
            # Try local backup
            self.save_local_backup()

    def save_local_backup(self):
        """Fallback to save locally if Drive fails"""
        try:
            local_file = f'local_backup_{self.session_id}.csv'
            df = pd.DataFrame(self.all_jobs_data)
            df.to_csv(local_file, index=False)
            print(f"    💾 Local backup saved: {local_file}")
        except Exception as e:
            print(f"    ❌ Local backup also failed: {e}")

    def load_checkpoint(self):
        """Load existing checkpoint if available (for resume)"""
        if os.path.exists(self.checkpoint_file):
            try:
                with open(self.checkpoint_file, 'rb') as f:
                    checkpoint_data = pickle.load(f)

                self.all_jobs_data = checkpoint_data['all_jobs_data']
                self.seen_job_ids = checkpoint_data['seen_job_ids']
                self.search_metadata = defaultdict(lambda: defaultdict(int),
                                                 checkpoint_data['search_metadata'])

                print(f"✅ Resumed from checkpoint: {len(self.all_jobs_data)} jobs loaded")
                print(f"   Last saved: {checkpoint_data['timestamp']}")

            except Exception as e:
                print(f"⚠️ Could not load checkpoint: {e}")

    def auto_checkpoint(self):
        """Automatically save checkpoint at intervals"""
        current_time = time.time()
        if current_time - self.last_checkpoint_time > self.checkpoint_interval:
            self.save_checkpoint()
            self.last_checkpoint_time = current_time

    def setup_driver(self, headless=False):
        """Setup Chrome driver"""
        from selenium.webdriver.chrome.options import Options

        chrome_options = Options()
        chrome_options.add_argument("--disable-blink-features=AutomationControlled")
        chrome_options.add_experimental_option("excludeSwitches", ["enable-automation"])
        chrome_options.add_argument("--no-sandbox")
        chrome_options.add_argument("--disable-dev-shm-usage")
        chrome_options.add_argument("--window-size=1920,1080")

        prefs = {"profile.managed_default_content_settings.images": 2}
        chrome_options.add_experimental_option("prefs", prefs)

        if headless:
            chrome_options.add_argument("--headless=new")

        self.driver = webdriver.Chrome(options=chrome_options)
        print("✅ Driver setup complete")

    def extract_jobs_working(self):
        """Your existing extraction method"""
        # [Keep your existing extraction code exactly as is]
        jobs = []

        try:
            time.sleep(3)

            job_links = self.driver.find_elements(By.CSS_SELECTOR,
                "a.base-card__full-link[href*='/jobs/view/']")

            base_cards = self.driver.find_elements(By.CSS_SELECTOR,
                "div.base-search-card.job-search-card")

            if len(job_links) < 10:
                print(f"    Found {len(job_links)} job links, {len(base_cards)} job cards")

            for i, link in enumerate(job_links):
                try:
                    job_data = {}

                    href = link.get_attribute('href')
                    if href:
                        job_id_match = re.search(r'/view/(\d+)', href)
                        if job_id_match:
                            job_data['job_id'] = job_id_match.group(1)
                        else:
                            job_data['job_id'] = f"job_{i}_{datetime.now().timestamp()}"

                        clean_url = href.split('?')[0]
                        job_data['url'] = clean_url
                    else:
                        continue

                    if job_data['job_id'] in self.seen_job_ids:
                        continue

                    title = link.text.strip()
                    if not title:
                        try:
                            span = link.find_element(By.CSS_SELECTOR, "span")
                            title = span.text.strip()
                        except:
                            pass

                    if not title:
                        title = link.get_attribute('aria-label')

                    job_data['title'] = title

                    if not job_data['title'] or len(job_data['title']) < 5:
                        continue
                    if any(skip in job_data['title'].lower() for skip in
                           ['promoted', 'sign in', 'join now', 'see more']):
                        continue

                    if i < len(base_cards):
                        card = base_cards[i]

                        try:
                            company_elem = card.find_element(By.TAG_NAME, "h4")
                            job_data['company'] = company_elem.text.strip()
                        except:
                            try:
                                company_elem = card.find_element(By.CSS_SELECTOR, ".base-search-card__subtitle")
                                job_data['company'] = company_elem.text.strip()
                            except:
                                job_data['company'] = 'N/A'

                        try:
                            location_elem = card.find_element(By.CSS_SELECTOR,
                                ".job-search-card__location")
                            job_data['location'] = location_elem.text.strip()
                        except:
                            job_data['location'] = 'N/A'

                        try:
                            date_elem = card.find_element(By.CSS_SELECTOR,
                                ".job-search-card__listdate, time")
                            job_data['posted_text'] = date_elem.text.strip()
                        except:
                            job_data['posted_text'] = 'N/A'
                    else:
                        job_data['company'] = 'N/A'
                        job_data['location'] = 'N/A'
                        job_data['posted_text'] = 'N/A'

                    job_data['scraped_at'] = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

                    jobs.append(job_data)
                    self.seen_job_ids.add(job_data['job_id'])

                except Exception as e:
                    continue

            if len(jobs) == 0:
                print(f"    ⚠️ No jobs extracted from page")
            elif len(jobs) < 5:
                print(f"    📊 Extracted {len(jobs)} jobs from page")

        except Exception as e:
            print(f"    ❌ Error in extraction: {str(e)}")

        return jobs

    def perform_stratified_random_sampling(self, target_total=10000):
        """
        Enhanced sampling with auto-save
        """
        print("\n" + "="*60)
        print("BULLETPROOF STRATIFIED RANDOM SAMPLING")
        print(f"Session ID: {self.session_id}")
        print(f"Auto-save to: {self.session_folder}")
        print("="*60)

        # Check if resuming
        if self.all_jobs_data:
            print(f"📂 Resuming from {len(self.all_jobs_data)} existing jobs")
            all_jobs = self.all_jobs_data
        else:
            all_jobs = []

        sampling_log = []

        city_targets = {}
        for city_key, config in self.cities_config.items():
            city_targets[city_key] = int(target_total * config['weight'])
            print(f"{config['name']}: {city_targets[city_key]:,} samples (weight: {config['weight']*100:.0f}%)")

        print("\n" + "-"*60)

        for city_key, config in self.cities_config.items():
            target_samples = city_targets[city_key]

            # Count existing jobs for this city
            existing_city_jobs = [j for j in all_jobs if j.get('search_city') == config['name']]
            city_jobs = existing_city_jobs.copy()

            if len(city_jobs) >= target_samples:
                print(f"\n📍 {config['name']} - Already complete ({len(city_jobs)} jobs)")
                continue

            print(f"\n📍 {config['name']} - Target: {target_samples:,} jobs (have {len(city_jobs)})")

            sampling_sequence = self.create_random_sequence(
                config['locations'],
                self.generic_search_terms,
                self.experience_levels,
                target_samples
            )

            searches_performed = 0
            max_searches = min(len(sampling_sequence), (target_samples - len(city_jobs)) // 10)

            for search_params in sampling_sequence[:max_searches]:
                if len(city_jobs) >= target_samples:
                    break

                location = search_params['location']
                term = search_params['term']
                exp_level = search_params['exp_level']

                print(f"\n  🔍 Search {searches_performed + 1}: '{term}' in {location}")

                search_url = self.build_search_url(term, location, exp_level)

                search_total_jobs = 0
                pages_to_scrape = 3

                for page_num in range(pages_to_scrape):
                    if len(city_jobs) >= target_samples:
                        break

                    if page_num == 0:
                        self.driver.get(search_url)
                    else:
                        page_url = f"{search_url}&start={page_num * 25}"
                        self.driver.get(page_url)
                        print(f"    📄 Page {page_num + 1}...")

                    time.sleep(random.uniform(3, 5))

                    try:
                        WebDriverWait(self.driver, 10).until(
                            EC.presence_of_element_located((By.CSS_SELECTOR, "a.base-card__full-link"))
                        )
                    except:
                        if page_num == 0:
                            print("    ⚠️ No results found")
                        break

                    page_jobs = self.extract_jobs_working()

                    if len(page_jobs) > 0:
                        remaining_needed = target_samples - len(city_jobs)
                        if len(page_jobs) > remaining_needed:
                            page_jobs = random.sample(page_jobs, remaining_needed)

                        for job in page_jobs:
                            job['search_city'] = config['name']
                            job['search_location'] = location
                            job['search_term'] = term
                            job['search_exp_level'] = exp_level['name']
                            job['search_page'] = page_num + 1
                            job['sampling_weight'] = 1.0 / config['weight']

                        city_jobs.extend(page_jobs)
                        all_jobs.extend(page_jobs)
                        search_total_jobs += len(page_jobs)

                    if page_num < pages_to_scrape - 1:
                        time.sleep(random.uniform(2, 4))

                print(f"    ✓ Got {search_total_jobs} jobs from this search")
                print(f"    Total collected: {len(city_jobs)}/{target_samples}")

                # CRITICAL: Update master list and auto-checkpoint
                self.all_jobs_data = all_jobs
                self.auto_checkpoint()

                sampling_log.append({
                    'city': config['name'],
                    'location': location,
                    'term': term,
                    'exp_level': exp_level['name'],
                    'jobs_found': search_total_jobs,
                    'pages_scraped': min(pages_to_scrape, page_num + 1),
                    'timestamp': datetime.now().isoformat()
                })

                searches_performed += 1

                delay = random.uniform(5, 10)
                print(f"    Waiting {delay:.1f} seconds...")
                time.sleep(delay)

                if searches_performed % 10 == 0:
                    print("  Taking extended break (30 seconds)...")
                    print("  Forcing checkpoint save...")
                    self.save_checkpoint()
                    time.sleep(30)

            print(f"\n  ✅ {config['name']} complete: {len(city_jobs):,} jobs collected")

        # Final save
        self.all_jobs_data = all_jobs
        self.save_checkpoint()

        return all_jobs, sampling_log

    def create_random_sequence(self, locations, terms, exp_levels, target_size):
        """Create randomized sequence of search parameters"""
        sequence = []

        for location in locations:
            for term in terms:
                for exp_level in exp_levels:
                    sequence.append({
                        'location': location,
                        'term': term,
                        'exp_level': exp_level
                    })

        random.shuffle(sequence)

        if len(sequence) < target_size // 20:
            sequence = sequence * (target_size // (20 * len(sequence)) + 1)
            random.shuffle(sequence)

        return sequence

    def build_search_url(self, search_term, location, exp_level):
        """Build LinkedIn search URL with parameters"""
        base_url = "https://www.linkedin.com/jobs/search?"

        params = [
            f"keywords={search_term.replace(' ', '%20')}",
            f"location={location.replace(' ', '%20').replace(',', '%2C')}",
            f"f_TPR={self.time_filter}",
            exp_level['param'],
            "sortBy=DD"
        ]

        return base_url + "&".join(params)

    def determine_category(self, title):
        """Categorize job based on title"""
        title_lower = title.lower()

        # [Keep your existing categorization logic]
        if any(term in title_lower for term in ['machine learning', 'ml engineer', 'ai engineer']):
            return 'AI/ML Engineering'
        elif any(term in title_lower for term in ['data scientist', 'data science']):
            return 'Data Science'
        elif any(term in title_lower for term in ['data engineer', 'etl developer']):
            return 'Data Engineering'
        elif any(term in title_lower for term in ['devops', 'site reliability', 'sre']):
            return 'DevOps/Infrastructure'
        elif any(term in title_lower for term in ['security engineer', 'cybersecurity']):
            return 'Security'
        elif any(term in title_lower for term in ['frontend', 'front-end']):
            return 'Frontend'
        elif any(term in title_lower for term in ['backend', 'back-end']):
            return 'Backend'
        elif any(term in title_lower for term in ['full stack', 'fullstack']):
            return 'Full Stack'
        elif any(term in title_lower for term in ['software engineer', 'software developer']):
            return 'Software Engineering (General)'
        else:
            return 'Other Technical'

    def save_final_results(self):
        """Save final results to Google Drive with proper error handling"""
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

        # Save to Drive
        drive_files = {
            'jobs': f'{self.session_folder}/final_jobs_{timestamp}.csv',
            'stats': f'{self.session_folder}/final_stats_{timestamp}.json',
            'summary': f'{self.session_folder}/final_summary_{timestamp}.json'
        }

        # Also save locally as backup
        local_files = {
            'jobs': f'final_jobs_{timestamp}.csv',
            'stats': f'final_stats_{timestamp}.json',
            'summary': f'final_summary_{timestamp}.json'
        }

        try:
            # Create DataFrame
            df = pd.DataFrame(self.all_jobs_data)
            df = df.drop_duplicates(subset=['job_id'], keep='first')

            # Categorize jobs
            df['category'] = df['title'].apply(self.determine_category)

            # Save to Google Drive
            df.to_csv(drive_files['jobs'], index=False)
            print(f"✅ Saved to Drive: {drive_files['jobs']}")

            # Also save locally
            df.to_csv(local_files['jobs'], index=False)
            print(f"✅ Local backup: {local_files['jobs']}")

            # Create summary
            summary = {
                'total_jobs': len(df),
                'unique_companies': df['company'].nunique(),
                'unique_titles': df['title'].nunique(),
                'categories': df['category'].value_counts().to_dict(),
                'cities': df['search_city'].value_counts().to_dict() if 'search_city' in df.columns else {},
                'session_id': self.session_id,
                'timestamp': timestamp
            }

            # Save summary to both locations
            with open(drive_files['summary'], 'w') as f:
                json.dump(summary, f, indent=2)

            with open(local_files['summary'], 'w') as f:
                json.dump(summary, f, indent=2)

            print("\n" + "="*60)
            print("✅ ALL DATA SAVED SUCCESSFULLY!")
            print(f"📁 Google Drive: {self.session_folder}")
            print(f"📁 Local Backup: final_jobs_{timestamp}.csv")
            print("="*60)

            return drive_files, local_files

        except Exception as e:
            print(f"❌ Error saving final results: {e}")
            print("Attempting emergency save...")

            # Emergency save - just dump what we have
            emergency_file = f'emergency_save_{self.session_id}.pkl'
            with open(emergency_file, 'wb') as f:
                pickle.dump(self.all_jobs_data, f)
            print(f"💾 Emergency save: {emergency_file}")

            return None, {'emergency': emergency_file}

    def close(self):
        """Close browser and ensure final save"""
        print("\n🏁 Finalizing...")

        # Final save before closing
        self.save_checkpoint()
        self.save_final_results()

        if self.driver:
            self.driver.quit()
            print("✅ Browser closed")


def run_bulletproof_analysis(target_jobs=10000):
    """
    Run bulletproof scraping with Google Drive backup
    """

    print("="*80)
    print("BULLETPROOF TECH JOB DEMAND ANALYSIS")
    print("With Google Drive Auto-Save - Never Lose Data Again!")
    print("="*80)

    # Check Drive is mounted
    if not os.path.exists('/content/drive/MyDrive'):
        print("❌ Google Drive not mounted!")
        print("Please run:")
        print("  from google.colab import drive")
        print("  drive.mount('/content/drive')")
        return None

    print("✅ Google Drive mounted")

    # Setup
    import subprocess
    import sys

    print("\n📦 Installing requirements...")
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'selenium', 'pandas', 'scipy'], capture_output=True)

    print("🔧 Setting up Chrome driver...")
    subprocess.run(['apt-get', 'update'], capture_output=True)
    subprocess.run(['apt', 'install', 'chromium-chromedriver'], capture_output=True)
    sys.path.insert(0, '/usr/lib/chromium-browser/chromedriver')

    # Initialize scraper
    scraper = BulletproofDemandScraper(drive_folder='linkedin_scraper_data')

    print("\n⚙️ Configuration:")
    print(f"  • Target: {target_jobs:,} jobs")
    print(f"  • Session ID: {scraper.session_id}")
    print(f"  • Auto-save every: 5 minutes")
    print(f"  • Backup location: {scraper.session_folder}")

    try:
        scraper.setup_driver(headless=True)

        # Run sampling
        print("\n🎲 Starting bulletproof sampling...")
        jobs, sampling_log = scraper.perform_stratified_random_sampling(target_total=target_jobs)

        # Save final results
        print("\n💾 Saving final results...")
        drive_files, local_files = scraper.save_final_results()

        print("\n" + "="*80)
        print("✅ COMPLETE - DATA SAFELY STORED!")
        print("="*80)

        return pd.DataFrame(jobs)

    except KeyboardInterrupt:
        print("\n⚠️ Interrupted by user - saving progress...")
        scraper.save_checkpoint()
        scraper.save_final_results()
        print("✅ Progress saved - can resume later")

    except Exception as e:
        print(f"\n❌ Error: {e}")
        print("💾 Attempting to save data before exit...")
        scraper.save_checkpoint()
        scraper.save_final_results()
        import traceback
        traceback.print_exc()

    finally:
        scraper.close()
        print("\n🏁 Scraper closed - all data saved")


# Usage
if __name__ == "__main__":
    df = run_bulletproof_analysis(target_jobs=10000)

# Analysis of jobs

In [3]:
"""
Tech Job Market Analysis
Objective analysis of 8,316 tech job postings
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter, defaultdict
import re
from datetime import datetime

class JobMarketAnalyzer:
    def __init__(self, data_path=None):
        """Initialize analyzer with data path"""
        self.df = None
        self.data_path = data_path or self._get_default_path()

    def _get_default_path(self):
        """Try to find data file in expected locations"""
        paths = [
            '/content/drive/MyDrive/linkedin_scraper_data/session_20250825_143043/final_jobs_20250825_235525.csv',
            'final_jobs_20250825_235525.csv'
        ]
        return paths

    def load_data(self):
        """Load job data from file"""
        if isinstance(self.data_path, list):
            for path in self.data_path:
                try:
                    self.df = pd.read_csv(path)
                    print(f"✓ Loaded {len(self.df):,} jobs from {path}")
                    return True
                except:
                    continue
            print("✗ Could not load data from any path")
            return False
        else:
            try:
                self.df = pd.read_csv(self.data_path)
                print(f"✓ Loaded {len(self.df):,} jobs")
                return True
            except:
                print(f"✗ Could not load data from {self.data_path}")
                return False

    def categorize_jobs(self):
        """Apply job categorization with clear hierarchy"""

        def categorize(title):
            title_lower = title.lower()

            # Executive/C-Suite (check first for hierarchy)
            if any(term in title_lower for term in ['chief', 'cto', 'cfo', 'coo', 'cpo', 'ciso', 'ceo']):
                return 'C-Suite'
            elif 'vp' in title_lower or 'vice president' in title_lower or 'svp' in title_lower:
                return 'Vice President'
            elif 'head of' in title_lower:
                return 'Head of Department'
            elif 'director' in title_lower and 'director of' in title_lower:
                return 'Director'

            # Management roles
            elif any(term in title_lower for term in ['engineering manager', 'tech lead', 'technical lead',
                                                      'team lead', 'development manager']):
                return 'Engineering Management'
            elif 'manager' in title_lower:
                return 'Management (Other)'

            # Product & Program
            elif any(term in title_lower for term in ['product manager', 'product owner']):
                return 'Product Management'
            elif any(term in title_lower for term in ['program manager', 'project manager']):
                if 'technical' in title_lower:
                    return 'Technical Program Management'
                return 'Project Management'

            # AI/ML & Data
            elif any(term in title_lower for term in ['machine learning', 'ml engineer', 'ai engineer',
                                                      'artificial intelligence', 'mlops']):
                return 'AI/ML Engineering'
            elif 'data scientist' in title_lower:
                return 'Data Science'
            elif any(term in title_lower for term in ['data engineer', 'etl developer', 'data pipeline']):
                return 'Data Engineering'
            elif any(term in title_lower for term in ['data analyst', 'business analyst', 'analytics']):
                return 'Data Analytics/BI'

            # Engineering specializations
            elif any(term in title_lower for term in ['devops', 'site reliability', 'sre', 'platform engineer']):
                return 'DevOps/Infrastructure'
            elif any(term in title_lower for term in ['security engineer', 'cybersecurity']):
                return 'Security'
            elif any(term in title_lower for term in ['frontend', 'front-end', 'react developer']):
                return 'Frontend'
            elif any(term in title_lower for term in ['backend', 'back-end']):
                return 'Backend'
            elif any(term in title_lower for term in ['full stack', 'fullstack']):
                return 'Full Stack'
            elif any(term in title_lower for term in ['mobile', 'ios', 'android']):
                return 'Mobile'
            elif any(term in title_lower for term in ['qa', 'quality assurance', 'test engineer']):
                return 'QA/Testing'

            # Architecture & Solutions
            elif 'architect' in title_lower:
                return 'Technical Architect'
            elif 'solutions engineer' in title_lower:
                return 'Solutions Engineering'

            # Design
            elif any(term in title_lower for term in ['designer', 'ux', 'ui']):
                return 'Design/UX'

            # Research & Science
            elif any(term in title_lower for term in ['scientist', 'researcher', 'research']):
                if any(bio in title_lower for bio in ['bio', 'clinical', 'pharma', 'chemistry']):
                    return 'Life Sciences'
                return 'Research'

            # Generic Engineering
            elif any(term in title_lower for term in ['software engineer', 'software developer', 'engineer']):
                return 'Software Engineering'

            # Support & Operations
            elif any(term in title_lower for term in ['it support', 'help desk', 'it specialist']):
                return 'IT Support'
            elif 'operations' in title_lower:
                return 'Operations'

            # Non-technical
            elif any(term in title_lower for term in ['recruiter', 'talent', 'sales', 'customer success',
                                                      'marketing', 'content', 'administrative']):
                return 'Non-Technical'

            else:
                return 'Uncategorized'

        self.df['category'] = self.df['title'].apply(categorize)

    def determine_seniority(self):
        """Determine seniority level from job title"""

        def get_seniority(title):
            title_lower = title.lower()

            if any(term in title_lower for term in ['intern', 'internship']):
                return 'Intern'
            elif any(term in title_lower for term in ['junior', 'jr.', 'entry level', 'new grad', 'associate']):
                return 'Junior'
            elif any(term in title_lower for term in ['senior', 'sr.', 'lead']):
                return 'Senior'
            elif any(term in title_lower for term in ['staff', 'principal', 'distinguished']):
                return 'Staff+'
            elif any(term in title_lower for term in ['manager', 'director', 'vp', 'head', 'chief']):
                return 'Management'
            else:
                return 'Mid-Level'

        self.df['seniority'] = self.df['title'].apply(get_seniority)

    def print_basic_stats(self):
        """Print dataset overview"""
        print("\n" + "="*60)
        print("DATASET OVERVIEW")
        print("="*60)

        print(f"Total jobs: {len(self.df):,}")
        print(f"Unique companies: {self.df['company'].nunique():,}")
        print(f"Unique job titles: {self.df['title'].nunique():,}")
        print(f"Date range: Jobs posted in last 7 days")

        if 'search_city' in self.df.columns:
            print(f"\nGeographic distribution:")
            for city, count in self.df['search_city'].value_counts().items():
                print(f"  {city}: {count:,} jobs ({count/len(self.df)*100:.1f}%)")

    def analyze_categories(self):
        """Analyze job category distribution"""
        print("\n" + "="*60)
        print("JOB CATEGORY ANALYSIS")
        print("="*60)

        category_counts = self.df['category'].value_counts()

        print(f"\nTop 15 categories by job count:")
        for i, (cat, count) in enumerate(category_counts.head(15).items(), 1):
            pct = count/len(self.df)*100
            print(f"{i:2}. {cat:30} {count:4} jobs ({pct:5.1f}%)")

        # Group analysis
        print("\n" + "-"*40)
        print("CATEGORY GROUPINGS")
        print("-"*40)

        groups = {
            'Leadership': ['C-Suite', 'Vice President', 'Director', 'Head of Department',
                          'Engineering Management', 'Management (Other)'],
            'IC Engineering': ['Software Engineering', 'AI/ML Engineering', 'Frontend',
                             'Backend', 'Full Stack', 'DevOps/Infrastructure', 'Security', 'Mobile'],
            'Data Roles': ['Data Science', 'Data Engineering', 'Data Analytics/BI'],
            'Product/Program': ['Product Management', 'Technical Program Management', 'Project Management'],
            'Other Technical': ['QA/Testing', 'Technical Architect', 'Solutions Engineering', 'Design/UX'],
            'Non-Technical': ['Non-Technical', 'IT Support', 'Operations', 'Life Sciences']
        }

        for group_name, categories in groups.items():
            group_jobs = self.df[self.df['category'].isin(categories)]
            count = len(group_jobs)
            pct = count/len(self.df)*100
            print(f"\n{group_name}: {count:,} jobs ({pct:.1f}%)")

            # Show breakdown
            for cat in categories:
                cat_count = len(self.df[self.df['category'] == cat])
                if cat_count > 0:
                    cat_pct = cat_count/len(self.df)*100
                    print(f"  - {cat}: {cat_count} ({cat_pct:.1f}%)")

    def analyze_seniority(self):
        """Analyze seniority distribution"""
        print("\n" + "="*60)
        print("SENIORITY ANALYSIS")
        print("="*60)

        seniority_counts = self.df['seniority'].value_counts()

        print("\nOverall seniority distribution:")
        for level, count in seniority_counts.items():
            pct = count/len(self.df)*100
            print(f"  {level:15} {count:5} jobs ({pct:5.1f}%)")

        # Seniority by category
        print("\n" + "-"*40)
        print("ENTRY-LEVEL OPPORTUNITIES BY CATEGORY")
        print("-"*40)

        junior_by_cat = self.df[self.df['seniority'].isin(['Junior', 'Intern'])].groupby('category').size()
        junior_by_cat = junior_by_cat.sort_values(ascending=False).head(10)

        print("\nCategories with most junior/intern positions:")
        for cat, count in junior_by_cat.items():
            total_in_cat = len(self.df[self.df['category'] == cat])
            pct_of_cat = count/total_in_cat*100 if total_in_cat > 0 else 0
            print(f"  {cat:30} {count:3} jobs ({pct_of_cat:4.1f}% of category)")

    def analyze_companies(self):
        """Analyze top hiring companies"""
        print("\n" + "="*60)
        print("COMPANY ANALYSIS")
        print("="*60)

        # Clean company data
        df_clean = self.df[self.df['company'] != 'N/A'].copy()

        top_companies = df_clean['company'].value_counts().head(20)

        print("\nTop 20 companies by total job postings:")
        for i, (company, count) in enumerate(top_companies.items(), 1):
            # Get category distribution for this company
            company_df = df_clean[df_clean['company'] == company]
            top_cat = company_df['category'].value_counts().index[0]
            top_cat_count = company_df['category'].value_counts().values[0]
            top_cat_pct = top_cat_count/count*100

            print(f"{i:2}. {company:25} {count:3} jobs (top: {top_cat} {top_cat_pct:.0f}%)")

    def analyze_ai_ml_reality(self):
        """Deep dive into AI/ML job market"""
        print("\n" + "="*60)
        print("AI/ML MARKET ANALYSIS")
        print("="*60)

        ai_jobs = self.df[self.df['category'] == 'AI/ML Engineering']

        print(f"\nAI/ML Engineering jobs: {len(ai_jobs):,} ({len(ai_jobs)/len(self.df)*100:.1f}% of total)")

        if len(ai_jobs) > 0:
            # By location
            print("\nGeographic distribution of AI/ML jobs:")
            if 'search_city' in self.df.columns:
                for city in self.df['search_city'].unique():
                    city_ai = len(ai_jobs[ai_jobs['search_city'] == city])
                    city_total = len(self.df[self.df['search_city'] == city])
                    print(f"  {city}: {city_ai} jobs ({city_ai/city_total*100:.1f}% of local market)")

            # By seniority
            print("\nAI/ML seniority requirements:")
            for level, count in ai_jobs['seniority'].value_counts().items():
                print(f"  {level}: {count} ({count/len(ai_jobs)*100:.1f}%)")

            # Top companies
            print("\nTop companies hiring for AI/ML:")
            for company, count in ai_jobs['company'].value_counts().head(10).items():
                if company != 'N/A':
                    company_total = len(self.df[self.df['company'] == company])
                    pct = count/company_total*100
                    print(f"  {company}: {count} AI/ML jobs ({pct:.1f}% of their {company_total} total)")

    def find_anomalies(self):
        """Identify interesting patterns and anomalies"""
        print("\n" + "="*60)
        print("NOTABLE FINDINGS")
        print("="*60)

        findings = []

        # Check mobile market
        mobile_count = len(self.df[self.df['category'] == 'Mobile'])
        mobile_pct = mobile_count/len(self.df)*100
        if mobile_pct < 1:
            findings.append(f"Mobile development: Only {mobile_count} jobs ({mobile_pct:.2f}% of market)")

        # Check junior market
        junior_count = len(self.df[self.df['seniority'] == 'Junior'])
        junior_pct = junior_count/len(self.df)*100
        findings.append(f"Junior positions: {junior_count} jobs ({junior_pct:.1f}% of market)")

        # Leadership vs IC
        leadership = len(self.df[self.df['seniority'] == 'Management'])
        ic_engineering = len(self.df[self.df['category'].isin(['Software Engineering', 'Frontend',
                                                               'Backend', 'Full Stack'])])
        if leadership > 0 and ic_engineering > 0:
            ratio = leadership/ic_engineering
            findings.append(f"Management to IC ratio: {ratio:.2f}:1")

        # Title patterns
        senior_titles = len(self.df[self.df['title'].str.lower().str.contains('senior|sr\\.', na=False)])
        findings.append(f"'Senior' in title: {senior_titles} jobs ({senior_titles/len(self.df)*100:.1f}%)")

        for i, finding in enumerate(findings, 1):
            print(f"{i}. {finding}")

    def generate_summary(self):
        """Generate executive summary"""
        print("\n" + "="*60)
        print("EXECUTIVE SUMMARY")
        print("="*60)

        # Calculate key metrics
        total_jobs = len(self.df)
        leadership_jobs = len(self.df[self.df['seniority'] == 'Management'])
        junior_jobs = len(self.df[self.df['seniority'].isin(['Junior', 'Intern'])])
        ai_ml_jobs = len(self.df[self.df['category'] == 'AI/ML Engineering'])
        swe_jobs = len(self.df[self.df['category'] == 'Software Engineering'])

        print(f"""
Market Snapshot ({total_jobs:,} jobs analyzed):

1. Category Distribution:
   - Top category: {self.df['category'].value_counts().index[0]} ({self.df['category'].value_counts().values[0]:,} jobs)
   - Software Engineering: {swe_jobs:,} jobs ({swe_jobs/total_jobs*100:.1f}%)
   - AI/ML Engineering: {ai_ml_jobs:,} jobs ({ai_ml_jobs/total_jobs*100:.1f}%)

2. Seniority Breakdown:
   - Management roles: {leadership_jobs:,} ({leadership_jobs/total_jobs*100:.1f}%)
   - Junior/Intern roles: {junior_jobs:,} ({junior_jobs/total_jobs*100:.1f}%)
   - Senior+ roles: {len(self.df[self.df['seniority'].isin(['Senior', 'Staff+'])]):,} ({len(self.df[self.df['seniority'].isin(['Senior', 'Staff+'])])/total_jobs*100:.1f}%)

3. Key Observations:
   - Junior positions represent {junior_jobs/total_jobs*100:.1f}% of market
   - AI/ML is {ai_ml_jobs/total_jobs*100:.1f}% of market (not as dominant as headlines suggest)
   - Traditional software engineering remains largest technical category
        """)

    def run_full_analysis(self):
        """Run complete analysis pipeline"""
        if not self.load_data():
            return None

        # Apply categorizations
        self.categorize_jobs()
        self.determine_seniority()

        # Run analyses
        self.print_basic_stats()
        self.analyze_categories()
        self.analyze_seniority()
        self.analyze_companies()
        self.analyze_ai_ml_reality()
        self.find_anomalies()
        self.generate_summary()

        # Save enhanced dataset
        output_file = 'analyzed_jobs_clean.csv'
        self.df.to_csv(output_file, index=False)
        print(f"\n✓ Enhanced dataset saved to: {output_file}")

        return self.df


# Usage
if __name__ == "__main__":
    analyzer = JobMarketAnalyzer()
    df_analyzed = analyzer.run_full_analysis()

✓ Loaded 8,316 jobs from /content/drive/MyDrive/linkedin_scraper_data/session_20250825_143043/final_jobs_20250825_235525.csv

DATASET OVERVIEW
Total jobs: 8,316
Unique companies: 999
Unique job titles: 1,333
Date range: Jobs posted in last 7 days

Geographic distribution:
  SF Bay Area: 3,500 jobs (42.1%)
  New York: 1,349 jobs (16.2%)
  Austin: 1,249 jobs (15.0%)
  Seattle: 1,218 jobs (14.6%)
  Boston: 1,000 jobs (12.0%)

JOB CATEGORY ANALYSIS

Top 15 categories by job count:
 1. C-Suite                        1392 jobs ( 16.7%)
 2. Uncategorized                  1131 jobs ( 13.6%)
 3. Software Engineering           1087 jobs ( 13.1%)
 4. Management (Other)              775 jobs (  9.3%)
 5. Vice President                  754 jobs (  9.1%)
 6. AI/ML Engineering               489 jobs (  5.9%)
 7. Head of Department              388 jobs (  4.7%)
 8. Research                        284 jobs (  3.4%)
 9. Full Stack                      266 jobs (  3.2%)
10. Technical Architect         